In [8]:
import pandas as pd
import numpy as np
import yfinance as yf
from scipy.stats import norm
import os

In [9]:
#Downloading the ticket data
ticker = "NESTLEIND.NS"

if os.path.exists("nestleind_data.csv"):
    data = pd.read_csv("nestleind_data.csv", index_col=0, parse_dates=True)
    data["Close"] = pd.to_numeric(data["Close"], errors="coerce")
    data = data.dropna()
else:
    data = yf.download("NESTLEIND.NS", period="6mo", interval="1d")
    data.to_csv("nestleind_data.csv")

/var/folders/3t/3xjt3f516dx8r11ngt2k1w_m0000gn/T/ipykernel_61345/2828290593.py:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  data = pd.read_csv("nestleind_data.csv", index_col=0, parse_dates=True)


In [10]:
#Calculating the volatility
data["Log Return"] = np.log(data["Close"] / data["Close"].shift(1))
data = data.dropna()
voldaily = data["Log Return"].std()
vol = voldaily * np.sqrt(252)
print("Daily Volatility:", voldaily)
print("Annualized Historical Volatility:", vol)

Daily Volatility: 0.011524548186591932
Annualized Historical Volatility: 0.18294653084461593


In [11]:
#Defining the BSM functions
def black_scholes(S, K, T, r, sigma, option_type="call"):
    d1 = (np.log(S / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    if option_type == "call":
        return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    else:
        return K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)

In [ ]:
#Market Data as on 11th April 2026
#Options chains data based on NSE

SNAPSHOT_DATE = "2026-04-11"
S = 1249.30
r = 0.053
ticker_name = "NESTLEIND.NS"

rows = [
    ("28-Apr", 16, "ATM Call", 1250, 28.15),
    ("28-Apr", 16, "ATM Put",  1250, 24.60),
    ("28-Apr", 16, "OTM Call", 1350,  2.40),
    ("28-Apr", 16, "OTM Put",  1160,  4.25),
]

df = pd.DataFrame(rows, columns=["Expiry", "Days", "Option", "Strike", "Market Price"])
print(df)

   Expiry  Days    Option  Strike  Market Price
0  28-Apr    16  ATM Call    1250         28.15
1  28-Apr    16   ATM Put    1250         24.60
2  28-Apr    16  OTM Call    1350          2.40
3  28-Apr    16   OTM Put    1160          4.25


In [ ]:
bsm_prices = []
for i in range(len(df)):
    K = df.loc[i, "Strike"]
    T = df.loc[i, "Days"] / 365
    opt_type = df.loc[i, "Option"]
    if "Call" in opt_type:
        price = black_scholes(S, K, T, r, vol, "call")
    else:
        price = black_scholes(S, K, T, r, vol, "put")
    bsm_prices.append(price)

df["BSM Price"] = bsm_prices
df["Difference"] = df["Market Price"] - df["BSM Price"]
df["Spot Price (S)"] = S
df["Volatility (vol)"] = vol
df["Risk-Free Rate (r)"] = r

print("=" * 60)
print(f"Stock: NESTLEIND | Spot: ₹{S} | Vol: {round(vol*100, 2)}% | r: {r}")
print("=" * 60)
print(df[["Expiry", "Option", "Strike", "Market Price", "BSM Price", "Difference"]].to_string(index=False))

Stock: NESTLEIND | Spot: ₹1249.3 | Vol: 18.29% | r: 0.053
Expiry   Option  Strike  Market Price  BSM Price  Difference
28-Apr ATM Call    1250         28.15  20.192967    7.957033
28-Apr  ATM Put    1250         24.60  17.992228    6.607772
28-Apr OTM Call    1350          2.40   0.465241    1.934759
28-Apr  OTM Put    1160          4.25   0.394300    3.855700


In [ ]:
df.to_excel("NESTLEIND_BSM_Comparison.xlsx", index=False)
print("Saved as NESTLEIND_BSM_Comparison.xlsx")

✅ Saved as NESTLEIND_BSM_Comparison.xlsx
